In [1]:
import pandas as pd
import pyarrow
from pybroker import Alpaca
import os

In [2]:
alpaca = Alpaca(os.environ['ALPACA_API_KEY'], os.environ['ALPACA_SECRET_KEY'])

In [9]:
 #Reads and cleans the composition file

def load_sp500_components(path):
    df = pd.read_csv(path, sep=",")
    df["tickers_list"] = df["tickers"].str.split(",")
    df = df.explode("tickers_list")
    df["tickers_list"] = df["tickers_list"].str.strip()

    return df[["date", "tickers"]].dropna().reset_index(drop=True)



In [4]:
#Creates daily components list
def build_daily_components(df):
    # 1. Parse date
    df["date"] = pd.to_datetime(df["date"], dayfirst=True)

    # 2. Ordina
    df = df.sort_values("date")

    # 3. Crea calendario continuo
    all_days = pd.date_range(
        start=df["date"].min(),
        end=df["date"].max(),
        freq="D"
    )
    calendar = pd.DataFrame({"date": all_days})

    # 4. Raggruppa tickers per data (come prima)
    grouped = df.groupby("date")["tickers"].apply(list).reset_index(name="tickers")

    # 5. Merge-asof backward per riempire i buchi
    daily_comp = pd.merge_asof(
        calendar,
        grouped.sort_values("date"),
        on="date",
        direction="backward"
    )

    return daily_comp


In [47]:
import time
import pandas as pd

def download_hf_data(daily_comp, start_date, end_date, timeframe="1m", block_days=7):
    """
    Download high-frequency Alpaca data for all S&P500 components
    appearing in daily_comp between start_date and end_date.

    Parameters
    ----------
    daily_comp : DataFrame
        Must contain columns: ["date", "tickers"] where tickers is a list.
    start_date : str or Timestamp
    end_date   : str or Timestamp
    timeframe  : str
        e.g. "1m", "5m", "1h"
    block_days : int
        Size of each download block (default 30 days)

    Returns
    -------
    DataFrame
        Concatenated HF data for all blocks.
    """

    # 1. Convert date column
   daily_comp["date"] = pd.to_datetime(daily_comp["date"], format="%m/%d/%Y")

    # 2. Filter global range
    start_global = pd.Timestamp(start_date)
    end_global   = pd.Timestamp(end_date)

    daily_filtered = daily_comp[
        (daily_comp["date"] >= start_global) &
        (daily_comp["date"] < end_global)
    ]

    # 3. Extract sorted list of unique dates
    dates = daily_filtered["date"].sort_values().to_list()

    all_dfs = []
    n = len(dates)

    # 4. Loop over blocks
    for i in range(0, n, block_days):
        start = dates[i]
        end = start + pd.Timedelta(days=block_days)

        # Select block
        block = daily_filtered[
            (daily_filtered["date"] >= start) &
            (daily_filtered["date"] < end)
        ]

        if block.empty:
            print(f"[INFO] No data in block {start.date()} → {end.date()}, skipped.")
            continue

        # Extract tickers for this block
        tickers_block = list(map(str, sorted(set().union(*block["tickers"]))))


        print(f"\n[INFO] Downloading {len(tickers_block)} tickers | {start.date()} → {end.date()}")
        


        # Time the API call
        t0 = time.time()

        df = alpaca.query(
            tickers_block,
            start_date=start,
            end_date=end,
            timeframe=timeframe
        )

        elapsed = time.time() - t0
        print(f"[INFO] → Received {len(df)} rows in {elapsed:.2f} seconds")

        df["block_start"] = start
        all_dfs.append(df)

    # 5. Final concatenation
    full_df = pd.concat(all_dfs, ignore_index=True)

    return full_df




   


In [10]:
df= load_sp500_components("S&P 500 Historical Components & Changes(11-16-2025).csv")


In [11]:
df

,date,tickers
0,02/01/1996,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,02/01/1996,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,02/01/1996,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,02/01/1996,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,02/01/1996,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
...,...,...
1341690,11/11/2025,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
1341691,11/11/2025,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
1341692,11/11/2025,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
1341693,11/11/2025,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."


In [12]:
df2= build_daily_components(df)

In [13]:
df2

,date,tickers
0,1996-01-02,"[AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,A..."
1,1996-01-03,"[AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,A..."
2,1996-01-04,"[AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,A..."
3,1996-01-05,"[AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,A..."
4,1996-01-06,"[AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,A..."
...,...,...
10902,2025-11-07,"[A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,AD..."
10903,2025-11-08,"[A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,AD..."
10904,2025-11-09,"[A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,AD..."
10905,2025-11-10,"[A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,AD..."


In [15]:
lengths=df2["tickers"].apply(len)

print(lengths.describe())

count    10907.000000
mean       498.158797
std          5.376602
min        487.000000
25%        495.000000
50%        497.000000
75%        503.000000
max        507.000000
Name: tickers, dtype: float64


In [ ]:
mask_few = lengths < 488
df2.loc[mask_few, ["date", "tickers"]].head().describe()


In [48]:
full_df = download_hf_data(
    df2,
    start_date="2025-01-02",
    end_date="2025-12-31",
    timeframe="1m",
    block_days=7
)


[INFO] Downloading 1 tickers | 2025-01-02 → 2025-01-09
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:29 

[INFO] → Received 953079 rows in 88.90 seconds

[INFO] Downloading 1 tickers | 2025-01-09 → 2025-01-16
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:15 

[INFO] → Received 782121 rows in 74.83 seconds

[INFO] Downloading 1 tickers | 2025-01-16 → 2025-01-23
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:13 

[INFO] → Received 784154 rows in 73.19 seconds

[INFO] Downloading 1 tickers | 2025-01-23 → 2025-01-30
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:14 

[INFO] → Received 982306 rows in 73.64 seconds

[INFO] Downloading 1 tickers | 2025-01-30 → 2025-02-06
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:13 

[INFO] → Received 995112 rows in 73.32 seconds

[INFO] Downloading 1 tickers | 2025-02-06 → 2025-02-13
Tickers nel blocco: 1
Loading bar data...
Loaded bar data: 0:01:13 

[INFO] → Receiv

In [20]:
len(df2["tickers"][0])

487